# Benchmarking and Improving OCR System for Southeast Asian Languages

## Data Collection

This section describes the process of collecting the data from Wikipedia.

In [1]:
from article_pdf import ArticlePDF
from download import download_article_texts, download_article_pdfs, get_articles_by_language

### Collecting Articles in English

I chose 100 Wikipedia articles as my dataset. These 100 articles are a collection of the top 20 most viewed Wikipedia articles from 5 categories ([Wikipedia:Popular pages](https://en.wikipedia.org/wiki/Wikipedia:Popular_pages)). From this list, I collected the article's URL file path in English.

In [2]:
people = ['Donald_Trump', 'Elizabeth_II', 'Barack_Obama', 'Christiano_Ronaldo', 'Michael_Jackson', 'Elon_Musk', 'Lady_Gaga', 'Adolf_Hitler', 'Eminem', 'Lionel_Messi', 'Justin_Bieber', 'Freddie_Mercury', 'Kim_Kardashian', 'Johnny_Depp', 'Steve_Jobs', 'Dwayne_Johnson', 'Michael_Jordan', 'Taylor_Swift', 'Stephen_Hawking', 'Kanye_West']
countries = ['United_States', 'India', 'United_Kingdom', 'Canada', 'Australia', 'China', 'Russia', 'Japan', 'Germany', 'France', 'Singapore', 'Israel', 'Pakistan', 'Philippines', 'Brazil', 'Italy', 'Netherlands', 'New Zealand', 'Ukraine', 'Spain']
cities = ['New_York_City', 'London', 'Hong_Kong', 'Los_Angeles', 'Dubai', 'Washington,_D.C.', 'Paris', 'Chicago', 'Angelsberg', 'Mumbai', 'San_Francisco', 'Rome', 'Monaco', 'Toronto', 'Tokyo', 'Philadelphia', 'Machu_Picchu', 'Jerusalem', 'Amsterdam', 'Boston'] # Excluding Singapore since listed for countries
life = ['Cat', 'Dog', 'Animal', 'Lion', 'Coronavirus', 'Tiger', 'Human', 'Dinosaur', 'Elephant', 'Virus', 'Horse', 'Photosynthesis', 'Evolution', 'Apple', 'Bird', 'Mammal', 'Potato', 'Polar_bear', 'Shark', 'Snake']
structures = ['Taj_Mahal', 'Burj_Khalifa', 'Statue_of_Liberty', 'Great_Wall_of_China', 'Eiffel_Tower', 'Berlin_Wall', 'Stonehenge', 'Mount_Rushmore', 'Colosseum', 'Auschwitz_concentration_camp', 'Great_Pyramid_of_Giza', 'One_World_Trade_Center', 'Empire_State_Building', 'White_House', 'Petra', 'Large_Hadron_Collider', 'Hagia_Sophia', 'Golden_Gate_Bridge', 'Panama_Canal', 'Angkor_Wat'] # Excluding Machu Picchu since listed for cities

english_titles = people + countries + cities + life + structures

english_articles = []
for english_title in english_titles:
    url = f'https://en.wikipedia.org/wiki/{english_title}'
    article = ArticlePDF(english_title, english_title, url, 'en')
    english_articles.append(article)

### Collecting Articles in SEA Languages

Using the [MediaWiki Action API](https://www.mediawiki.org/wiki/API:Main_page), I then collected the articles' URLs and names in the following languages:
- Thai
- Vietnamese
- Bahasa Indonesian


I referred to the [Wikimedia Language Codes](https://www.wikidata.org/wiki/Help:Wikimedia_language_codes/lists/all) to identify the mapping of languages to language codes used by the API.

The API failed to fetch some English articles in other languages. The missing articles include:
- Thai:
    - Christiano_Ronaldo
    - Angelsberg
- Vietnamese:
    - Christiano_Ronaldo
    - Angelsberg
- Bahasa:
    - Christiano_Ronaldo
    - Angelsberg

In [3]:
thai_articles = get_articles_by_language(english_articles, 'th')
vietnamese_articles = get_articles_by_language(english_articles, 'vi')
bahasa_articles = get_articles_by_language(english_articles, 'id')

Issue fetching for Christiano_Ronaldo in th
Issue fetching for Angelsberg in th
Issue fetching for Christiano_Ronaldo in vi
Issue fetching for Angelsberg in vi
Issue fetching for Christiano_Ronaldo in id
Issue fetching for Angelsberg in id


### Downloading Article PDFs and Text

At this point, I had collected the following meta data for each article in each language:
- Article title
- Article title in English
- Article URL
- Language of article

Using [Selenium](https://selenium-python.readthedocs.io/), I then downloaded the articles in PDF format. These PDFs serve as my dataset of images to perform OCR on.

Lastly, using [MediaWiki Action API](https://www.mediawiki.org/wiki/API:Main_page), I downloaded the text of the articles into `.txt` files. These files serve as my ground truth.

Some things to note:
- Downloading the articles take around 10-15 minutes.
- You can change the destination folder to store the downloaded files by modifying the arguments passed into `download_article_pdfs` and `download_article_texts`. Currently, it's set to `data/<language>`.

In [4]:
download_article_pdfs(english_articles, 'data/english')
download_article_pdfs(thai_articles, 'data/thai')
download_article_pdfs(vietnamese_articles, 'data/vietnamese')
download_article_pdfs(bahasa_articles, 'data/bahasa')

download_article_texts(english_articles, 'data/english')
download_article_texts(thai_articles, 'data/thai')
download_article_texts(vietnamese_articles, 'data/vietnamese')
download_article_texts(bahasa_articles, 'data/bahasa')

## Data Preprocessing

This section describes the steps done to pre-process the collected data.

### Converting PDFs to Images

As most OCR tools do not read documents in PDF format, I converted the collected PDFs into PNG images. If the PDF file has multiple pages, then multiple PNG images are stored to represent each page.

In [2]:
from preprocess import convert_pdfs_to_pngs

convert_pdfs_to_pngs('data/english')
convert_pdfs_to_pngs('data/thai')
convert_pdfs_to_pngs('data/vietnamese')
convert_pdfs_to_pngs('data/bahasa')

## Benchmarking

### EasyOCR

In [1]:
import easyocr

# This needs to run only once to load the model into memory
en_reader = easyocr.Reader(['en']) 
th_reader = easyocr.Reader(['th'])
vi_reader = easyocr.Reader(['vi'])
id_reader = easyocr.Reader(['id'])

In [22]:
import os
import time
from typing import List

NUM_OF_ARTICLES = 20

def run_easy_ocr(source_path: str, reader):
    count = 0
    for f in os.listdir(source_path):
        if count >= NUM_OF_ARTICLES:
            break
        count += 1

        print(f'Running on article: {f}')
        i = 0
        image_file_path = f'{source_path}/{f}/page-{i}.png'
        res = ''
        while os.path.exists(image_file_path):
            text = reader.readtext(image_file_path, detail=0)
            res += ' '.join(text)
            i += 1
            image_file_path = f'{source_path}/{f}/page-{i}.png'
        with open(f'{source_path}/{f}/easy-ocr-results.txt', 'wt') as outfile:
            outfile.write(res)

def run_easy_ocr_on_articles(articles: List[str], source_path: str, reader):
    for article in articles:
        start_time = time.time()
        print(f'Running on article: {article}')

        result_file_path = f'{source_path}/{article}/easy-ocr-results.txt'
        if os.path.exists(result_file_path):
            print('Results exist')
            continue

        res = ''
        i = 0
        image_file_path = f'{source_path}/{article}/page-{i}.png'
        while os.path.exists(image_file_path):
            text = reader.readtext(image_file_path, detail=0)
            res += ' '.join(text)
            i += 1
            image_file_path = f'{source_path}/{article}/page-{i}.png'
        with open(result_file_path, 'wt') as outfile:
            outfile.write(res)
        
        end_time = time.time()
        time_taken = end_time - start_time
        print(f'Finished! Time taken: {time_taken} seconds')

In [24]:
# Find shortest articles
import os

res = []

source_path = 'data/english'
for f in os.listdir(source_path):
    i = 0
    image_file_path = f'{source_path}/{f}/page-{i}.png'
    while os.path.exists(image_file_path):
        i += 1
        image_file_path = f'{source_path}/{f}/page-{i}.png'
    res.append((f, i))

articles_sorted_by_length = sorted(res, key=lambda x: x[1])
print(articles_sorted_by_length)

shortest_articles = list(map(lambda x: x[0], filter(lambda x: x[0] != 'Christiano_Ronaldo' and x[0] != 'Angelsberg', articles_sorted_by_length[:22])))
shortest_articles

[('Angelsberg', 1), ('Polar_bear', 2), ('Mount_Rushmore', 10), ('Potato', 12), ('Burj_Khalifa', 12), ('Machu_Picchu', 14), ('Petra', 14), ('Animal', 14), ('Great_Wall_of_China', 16), ('Angkor_Wat', 17), ('Taj_Mahal', 18), ('Colosseum', 20), ('Japan', 20), ('Christiano_Ronaldo', 20), ('Elizabeth_II', 21), ('Apple', 22), ('Photosynthesis', 22), ('Coronavirus', 23), ('Lionel_Messi', 24), ('Eiffel_Tower', 24), ('Large_Hadron_Collider', 24), ('Monaco', 25), ('White_House', 25), ('Amsterdam', 25), ('Tokyo', 26), ('Golden_Gate_Bridge', 26), ('Dubai', 26), ('Rome', 26), ('Mammal', 27), ('Hong_Kong', 27), ('Michael_Jordan', 28), ('Netherlands', 29), ('Boston', 29), ('Elephant', 29), ('Mumbai', 31), ('Dinosaur', 31), ('Virus', 31), ('Snake', 31), ('Ukraine', 32), ('Statue_of_Liberty', 32), ('Toronto', 32), ('Horse', 32), ('Lion', 32), ('Tiger', 33), ('Germany', 33), ('Stonehenge', 33), ('Freddie_Mercury', 33), ('New Zealand', 33), ('Shark', 33), ('Paris', 34), ('Dog', 34), ('Chicago', 34), ('Spa

['Polar_bear',
 'Mount_Rushmore',
 'Potato',
 'Burj_Khalifa',
 'Machu_Picchu',
 'Petra',
 'Animal',
 'Great_Wall_of_China',
 'Angkor_Wat',
 'Taj_Mahal',
 'Colosseum',
 'Japan',
 'Elizabeth_II',
 'Apple',
 'Photosynthesis',
 'Coronavirus',
 'Lionel_Messi',
 'Eiffel_Tower',
 'Large_Hadron_Collider',
 'Monaco']

In [35]:
source_path = 'data/bahasa'
res = []
for f in os.listdir(source_path):
    if f not in shortest_articles:
        continue
    i = 0
    image_file_path = f'{source_path}/{f}/page-{i}.png'
    while os.path.exists(image_file_path):
        i += 1
        image_file_path = f'{source_path}/{f}/page-{i}.png'
    res.append((f, i))
res

[('Apple', 12),
 ('Polar_bear', 3),
 ('Mount_Rushmore', 2),
 ('Colosseum', 4),
 ('Machu_Picchu', 3),
 ('Potato', 4),
 ('Photosynthesis', 7),
 ('Taj_Mahal', 2),
 ('Lionel_Messi', 17),
 ('Japan', 23),
 ('Coronavirus', 6),
 ('Eiffel_Tower', 13),
 ('Elizabeth_II', 26),
 ('Monaco', 8),
 ('Petra', 12),
 ('Great_Wall_of_China', 7),
 ('Large_Hadron_Collider', 6),
 ('Animal', 12),
 ('Angkor_Wat', 6),
 ('Burj_Khalifa', 4)]

In [23]:
# run_easy_ocr('data/thai', th_reader)
run_easy_ocr_on_articles(shortest_articles, 'data/thai', th_reader)

Running on article: Polar_bear
Results exist
Running on article: Mount_Rushmore
Results exist
Running on article: Potato
Finished! Time taken: 14.3244469165802 seconds
Running on article: Burj_Khalifa
Finished! Time taken: 24.016481161117554 seconds
Running on article: Machu_Picchu
Finished! Time taken: 17.165898084640503 seconds
Running on article: Petra
Finished! Time taken: 16.166284799575806 seconds
Running on article: Animal
Finished! Time taken: 47.08907413482666 seconds
Running on article: Great_Wall_of_China
Finished! Time taken: 19.416107892990112 seconds
Running on article: Angkor_Wat
Finished! Time taken: 72.4399082660675 seconds
Running on article: Taj_Mahal
Finished! Time taken: 28.660631895065308 seconds
Running on article: Colosseum
Finished! Time taken: 5.422324895858765 seconds
Running on article: Japan
Finished! Time taken: 227.48515605926514 seconds
Running on article: Elizabeth_II
Finished! Time taken: 122.99491000175476 seconds
Running on article: Apple
Results exi

In [27]:
# run_easy_ocr('data/english', en_reader)
# run_easy_ocr('data/vietnamese', vi_reader)
# run_easy_ocr('data/bahasa', id_reader)

# run_easy_ocr_on_articles(shortest_articles, 'data/english', en_reader)
# run_easy_ocr_on_articles(shortest_articles, 'data/vietnamese', vi_reader)
run_easy_ocr_on_articles(shortest_articles, 'data/bahasa', id_reader)

Running on article: Polar_bear
Finished! Time taken: 10.072195053100586 seconds
Running on article: Mount_Rushmore
Finished! Time taken: 6.677720785140991 seconds
Running on article: Potato
Finished! Time taken: 13.164318084716797 seconds
Running on article: Burj_Khalifa
Finished! Time taken: 7.71886420249939 seconds
Running on article: Machu_Picchu
Finished! Time taken: 7.411307096481323 seconds
Running on article: Petra
Finished! Time taken: 41.22079014778137 seconds
Running on article: Animal
Finished! Time taken: 77.12726211547852 seconds
Running on article: Great_Wall_of_China
Finished! Time taken: 24.797030925750732 seconds
Running on article: Angkor_Wat
Finished! Time taken: 24.021788120269775 seconds
Running on article: Taj_Mahal
Finished! Time taken: 4.959687948226929 seconds
Running on article: Colosseum
Finished! Time taken: 13.12365174293518 seconds
Running on article: Japan
Finished! Time taken: 81.3571708202362 seconds
Running on article: Elizabeth_II
Finished! Time taken

### Tesseract

In [28]:
import pytesseract

In [29]:
import os
import time
from typing import List

def run_tesseract(source_path: str, language: str):
    for f in os.listdir(source_path):
        print(f)
        i = 0
        image_file_path = f'{source_path}/{f}/page-{i}.png'
        res = ''
        while (os.path.exists(image_file_path)):
            text = pytesseract.image_to_string(image_file_path, lang=language)
            res += ' '.join(text)
            i += 1
            image_file_path = f'{source_path}/{f}/page-{i}.png'
        with open(f'{source_path}/{f}/tesseract-results.txt', 'wt') as outfile:
            outfile.write(res)

def run_tesseract_on_articles(articles: List[str], source_path: str, language: str):
    for article in articles:
        start_time = time.time()
        print(f'Running on article: {article}')

        result_file_path = f'{source_path}/{article}/tesseract-results.txt'
        if os.path.exists(result_file_path):
            print('Results exist')
            continue

        res = ''
        i = 0
        image_file_path = f'{source_path}/{article}/page-{i}.png'
        while os.path.exists(image_file_path):
            text = pytesseract.image_to_string(image_file_path, lang=language)            
            res += ' '.join(text)
            i += 1
            image_file_path = f'{source_path}/{article}/page-{i}.png'
        with open(result_file_path, 'wt') as outfile:
            outfile.write(res)
        
        end_time = time.time()
        time_taken = end_time - start_time
        print(f'Finished! Time taken: {time_taken} seconds')

In [32]:
# run_tesseract('data/thai', 'tha')
# run_tesseract_on_articles(shortest_articles, 'data/thai', 'tha')
run_tesseract_on_articles(shortest_articles, 'data/english', 'eng')
run_tesseract_on_articles(shortest_articles, 'data/vietnamese', 'vie')
run_tesseract_on_articles(shortest_articles, 'data/bahasa', 'ind')

Running on article: Polar_bear
Finished! Time taken: 1.6898150444030762 seconds
Running on article: Mount_Rushmore
Finished! Time taken: 25.245693922042847 seconds
Running on article: Potato
Finished! Time taken: 31.095876932144165 seconds
Running on article: Burj_Khalifa
Finished! Time taken: 35.45262694358826 seconds
Running on article: Machu_Picchu
Finished! Time taken: 31.326441764831543 seconds
Running on article: Petra
Finished! Time taken: 34.52344608306885 seconds
Running on article: Animal
Finished! Time taken: 40.00341296195984 seconds
Running on article: Great_Wall_of_China
Finished! Time taken: 24.967286109924316 seconds
Running on article: Angkor_Wat
Finished! Time taken: 31.587785005569458 seconds
Running on article: Taj_Mahal
Finished! Time taken: 33.6811089515686 seconds
Running on article: Colosseum
Finished! Time taken: 30.798343896865845 seconds
Running on article: Japan
Finished! Time taken: 344.9951889514923 seconds
Running on article: Elizabeth_II
Finished! Time t

In [24]:
source_path = 'data/thai'
i = 0
for f in os.listdir(source_path):
    file_path = f'{source_path}/{f}/tesseract-results.txt'
    if os.path.exists(file_path):
        i += 1
print(i)

# EasyOCR: 30 in 40 minutes
# Tesseract: 35 in 20 minutes

35


In [7]:
from torchmetrics.text import CharErrorRate
cer = CharErrorRate()

# predictions = ["this is the prediction", "there is an other sample"]
# targets = ["this is the reference", "there is another one"]
# print(cer(predictions, targets))

# # 0.0 CER
# predictions = ["100% match", "this is some text"]
# targets = ["100% match", "this is some text"]
# print(cer(predictions, targets))

source_path = 'data/vietnamese'

predictions = []
targets = []
# for f in os.listdir(source_path):
#     prediction_file_path = f'{source_path}/{f}/tesseract-results.txt'
#     target_file_path = f'{source_path}/{f}/text.txt'
#     try:
#         prediction_file = open(prediction_file_path, "r")
#         target_file = open(target_file_path, "r")
#         predictions.append(prediction_file.read())
#         targets.append(target_file.read())
#     except:
#         print('File does not exist')

t = ['Polar_bear', 'Mount_Rushmore', 'Potato', 'Burj_Khalifa', 'Machu_Picchu', 'Petra', 'Animal', 'Great_Wall_of_China', 'Taj_Mahal', 'Colosseum', 'Apple', 'Photosynthesis', 'Eiffel_Tower', 'Large_Hadron_Collider']

v = ['Mount_Rushmore', 'Petra', 'Animal', 'Colosseum', 'Photosynthesis', 'Coronavirus', 'Large_Hadron_Collider', 'Monaco']

b = ['Polar_bear', 'Mount_Rushmore', 'Potato', 'Burj_Khalifa', 'Machu_Picchu', 'Angkor_Wat', 'Taj_Mahal', 'Colosseum', 'Photosynthesis', 'Coronavirus', 'Large_Hadron_Collider', 'Monaco']

for article in v:
    prediction_file_path = f'{source_path}/{article}/easy-ocr-results.txt'
    prediction_file = open(prediction_file_path, "r")
    target_file_path = f'{source_path}/{article}/text.txt'
    target_file = open(target_file_path, "r") 

    r = cer(prediction_file.read(), target_file.read())
    print(f'{article}: {round(r.item(), 3)}')

        
# test_prediction = [predictions[0]]
# test_target = [targets[0]]
print('yay')
# print(cer(test_prediction, test_target))

Mount_Rushmore: 0.703
Petra: 0.362
Animal: 4.849
Colosseum: 0.481
Photosynthesis: 0.39
Coronavirus: 0.981
Large_Hadron_Collider: 0.649
Monaco: 0.509
yay
